In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

In [ ]:
sys.path.append('/home/evolvotron/code/pomap')

In [ ]:
import polars as pl

from core.interpreter import _fit, _collect_labels, _collect_leaves, _predict, _get_train_df_for_label, _get_test_df_for_label, _print_tree
from core.nodes import Lift, Ensemble, Leaf,  PomapNode, LearnsFrom
from core.label import Label
from dataclasses import dataclass
from typing import Iterator, Tuple, Callable, Iterable
from polars import DataFrame, Series
from core.interface import Model

import datetime as dt

### Set up some test models

In [ ]:
from scipy import stats

In [ ]:
@dataclass
class TestModel:
    x_column: str
    value = None
    hyperparameter = None

    def fit(self, df: pl.DataFrame):
        self.value = df[self.x_column].mean()

    def predict(self, df: pl.DataFrame):
        hyperparameter = self.hyperparameter or 0
        return [self.value + hyperparameter] * df.shape[0]


### Test Data

In [ ]:
times = pl.datetime_range(
    dt.datetime(2025, 1, 1),
    dt.datetime(2025, 1, 7),
    interval = '1d',
    eager=True
).rename('ts').to_frame()

df = times.with_columns(x = pl.arange(0, times.shape[0]))

# Example 1: Rolling retrain

In [ ]:
model = Leaf(
    label='tester',
    factory = lambda: TestModel(x_column='x')
    )

In [ ]:
retrain_points = [
    dt.datetime(2025, 1, 3),
    dt.datetime(2025, 1, 5)
]

rolling_train = Lift(
    model,
    atomics = retrain_points,
    train_mask_for_label = lambda label: pl.col('ts') <= label,
    test_mask_for_label = lambda label: pl.col('ts') > label,
    name='RollingRetrain'
)

In [ ]:
fitted_models, _ = _fit(rolling_train, df)
_predict(rolling_train, fitted_models, df)

In [ ]:
print(_print_tree(rolling_train))

### TODO Example 1.1 Rolling retrain with LGBM


In [ ]:
import lightgbm as lgbm

In [ ]:
lgmb_leaf = Leaf(
    label='lgbm',
    factory = lambda : lgbm.LGBMModel(objective='regression')
)

### TODO 

### Example 2: Randomised Cross Validation

In [ ]:
# For this one we need a bigger test dataframe
times = pl.datetime_range(
    dt.datetime(2025, 1, 1),
    dt.datetime(2025, 1, 7),
    interval = '1m',
    eager=True
).rename('ts').to_frame()

df = times.with_columns(x = pl.arange(0, times.shape[0]))

In [ ]:
model = Leaf(
    label='tester',
    factory = lambda: TestModel(x_column='x')
    )

In [ ]:
def random_assignment_expr(column, num_categories, seed=0) -> pl.Expr:
    return pl.col(column).hash(seed=seed).mod(num_categories)

train_mask_for_label = lambda fold_number: random_assignment_expr('ts', 5) == fold_number

randomised_cv = Lift(
    model,
    atomics = range(5),
    train_mask_for_label=train_mask_for_label,
    test_mask_for_label = lambda fold_number: ~train_mask_for_label(fold_number),
    name='RandomFold'
)

In [ ]:
for fold in range(5):
    label = Label(RandomFold=fold, leaf='tester')
    train_set_size =  _get_train_df_for_label(randomised_cv,  df, label).shape[0]/df.shape[0]
    test_set_size = _get_test_df_for_label(randomised_cv,  df, label).shape[0]/df.shape[0]
    print(f'Train Set Size: {train_set_size:.2f}, Test set size {test_set_size:.2f}'
    )

In [ ]:
models, _ = _fit(randomised_cv, df)

In [ ]:
for label, model in models.items():
    print(label['RandomFold'], model.value)
    

In [ ]:
_predict(randomised_cv, models, df).head()

In [ ]:
_predict(randomised_cv, models, df).select(pl.all().is_null().mean().round(2))

### Randomised CV with hyperparameter tuning

In [ ]:
import numpy as np

test_data = pl.datetime_range(
    dt.datetime(2025, 1, 1),
    dt.datetime(2025, 7, 1),
    interval='10m',
    eager=True
).rename('ts').to_frame()

test_data = test_data.with_columns(
    x=np.random.normal(1, 0.1, size=test_data.shape[0]),
    eps = np.random.normal(1, 0.05, size=test_data.shape[0]),
).with_columns(
    y = pl.col('x')*2 + pl.col('eps')
)


In [ ]:
cv_model = Leaf(
    label='cv_model',
    factory = lambda: TestModel(x_column='x', hyperparameter=0)
    )

In [ ]:
def random_assignment_expr(column, num_categories, seed=0) -> pl.Expr:
    return pl.col(column).hash(seed=seed).mod(num_categories)

in_train_period = pl.col('ts') <= dt.datetime(2025, 6, 1)
train_mask_for_label = lambda fold_number: (random_assignment_expr('ts', 5) == fold_number) & in_train_period

randomised_cv = Lift(
    cv_model,
    atomics = range(5),
    train_mask_for_label=train_mask_for_label,
    test_mask_for_label = lambda fold_number: ~train_mask_for_label(fold_number) & in_train_period,
    name='RandomFold'
)

In [ ]:
def logic(model: Model, df: pl.DataFrame):
    predictions = model.predict(df)
    res = {}

    for label in _collect_labels(model.root):
        eps_hat = predictions[label.column()] - predictions['y']
        res[label['RandomFold']] = eps_hat.mean()

    eps_hat = np.mean(list(res.values()))

    return {'hyperparameter': eps_hat}


In [ ]:
# 
test = Leaf(
    label='test',
    factory = lambda hyperparameter: TestModel(x_column='x', hyperparameter=hyperparameter)
    )

pipeline = LearnsFrom(learner=test, learns_from=randomised_cv, learn_logic=logic)

In [ ]:
_fit(pipeline, test_data)

### Example 4: Rolling retrain with hyperparameter tuning

### Example 3: Stateful Training